    Загрузка всех сделок из МТ
        Выделение Балансовых операций за вычетом реферальных
        Выделение реферальных операций
        Выделение торговых операций

    Сравнение Балансовых, реферальных, торговых операций на стороне MetaTrader 5 и CRM

In [1]:
import os
import sys
import importlib  # Добавляем импорт библиотеки для динамического импорта модулей
import pandas as pd
import datetime

import re
import numpy as np

import warnings
# Отключение предупреждения FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)

# Функция для динамического импорта модулей <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
sys.path.append(os.path.abspath("c:/unique_data/rep_fo_metatrader_server"))
file_imports = "dynamic_import_functions.py"  # Файл с функциями для динамического импорта

if os.path.exists(file_imports):
    importlib.invalidate_caches()           # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions  # Импортируем только нужные функции
    print(f"Импорт [{file_imports}] успешен.")
else: print(f"ERROR: Файл '{file_imports}' не найден, импорт не выполнен.")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

modules_to_import = {                                               # Словарь модулей для импорта
    "imports":
        ["c:/unique_data/rep_fo_metatrader_server",
                "pd_set_option",
                "mt5admin",
                "admin_connect",
                "admin_disconnect"],
    "yar_sed_general_lib":
        ["c:/unique_data/rep_fo_metatrader_server",
                "list_print",
                "int_list_to_csv_file",
                "load_account_list",
                "find_intersecting_lists"],
    "sed_array_lib":
        ["c:/unique_data/rep_fo_metatrader_server",
            "array_to_dataframe"]
                }
imported = import_functions(modules_to_import) # Импортируем модули из словаря modules_to_import

Импорт [dynamic_import_functions.py] успешен.
Импорт из 'imports' успешен: ['pd_set_option', 'mt5admin', 'admin_connect', 'admin_disconnect']
Импорт из 'yar_sed_general_lib' успешен: ['list_print', 'int_list_to_csv_file', 'load_account_list', 'find_intersecting_lists']
Импорт из 'sed_array_lib' успешен: ['array_to_dataframe']


In [2]:
# получить id CRM сделок к переоткрытию и проверить нет ли этих id уже на МТ5
account_id_list = imported["load_account_list"](r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250118\acc_list.csv")
imported["list_print"](account_id_list, "Счета к миграции")
admin =  imported["mt5admin"]()
admin_connect = imported["admin_connect"]()
if  admin_connect is not None:
    print("Подключен Администратор МТ5")
    deal_array = admin_connect.DealRequestByLoginsNumPy(account_id_list, 307718788, 1759407988)
    print(deal_array)
    mt5_deal_df = imported["array_to_dataframe"](deal_array)
    imported["admin_disconnect"]()
    del deal_array
else: print("Ошибка подключения")

imported["pd_set_option"]("Список всех сделок их МТ5", mt5_deal_df, 5)

Файл: C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250118\acc_list.csv; Длинна списка: 1548
[1548] элементов в списке [Счета к миграции] список: [262146, 262151, 73740, 229393, 237585, 237591, 237602, 163878, 229419, 237613, 221255, 262218, 237646, 237656, 237657, 114780, 237660, 237678, 65649, 237689, 262266, 73852, 237694, 90243, 262291, 262298, 262302, 237728, 262308, 262317, 262320, 237747, 237748, 262324, 237757, 262335, 237764, 262340, 180429, 237776, 237778, 237779, 237796, 237797, 262391, 262399, 262411, 237841, 262422, 262424, 196889, 237854, 147748, 237870, 237871, 237878, 237895, 237904, 262480, 262483, 262490, 82283, 180587, 237940, 262523, 262524, 57730, 74114, 262540, 262552, 164260, 262570, 262572, 238002, 238004, 246204, 238013, 254397, 262593, 254406, 262604, 262606, 254419, 238048, 238056, 238068, 205301, 238070, 262647, 238078, 262655, 254464, 238081, 238097, 262681, 115229, 139807, 115232, 238111, 238114, 41508, 254500, 238119, 238

,Deal (uint64),ExternalID (|S128),Login (uint64),Dealer (uint64),Order (uint64),Action (uint32),Entry (uint32),Digits (uint32),DigitsCurrency (uint32),ContractSize (float64),Time (int64),Symbol (|S128),Price (float64),Volume (uint64),Profit (float64),Storage (float64),Commission (float64),ObsoleteValue (float64),RateProfit (float64),RateMargin (float64),ExpertID (uint64),PositionID (uint64),Comment (|S128),ProfitRaw (float64),PricePosition (float64),VolumeClosed (uint64),TickValue (float64),TickSize (float64),Flags (uint64),TimeMsc (int64),Reason (uint32),Gateway (|S64),PriceGateway (float64),ModificationFlags (uint32),PriceSL (float64),PriceTP (float64),VolumeExt (uint64),VolumeClosedExt (uint64),Fee (float64),Value (float64),MarketBid (float64),MarketAsk (float64),MarketLast (float64)
0,3436674,b'',139807,1100,0,2,0,2,2,0.0,1688983353,b'',0.0,0,1000.0,0.0,0.0,0.0,0.0,0.0,0,0,"b'#42623, balance, deposit'",0.0,0.0,0,0.0,0.0,0,1688983353000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
1,3436675,b'',37788,1100,0,5,0,2,2,0.0,1688995885,b'',0.0,0,2341245.0,0.0,0.0,0.0,0.0,0.0,0,0,"b'#42625, balance, correction'",0.0,0.0,0,0.0,0.0,0,1688995885000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268707,3709171,b'',267230,1100,0,3,0,2,2,0.0,1736886647,b'',0.0,0,-50.0,0.0,0.0,0.0,0.0,0.0,0,0,"b'#61347, balance, credit'",0.0,0.0,0,0.0,0.0,0,1736886647000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0
268708,3709172,b'',263659,1100,0,3,0,2,2,0.0,1736936474,b'',0.0,0,-259.0,0.0,0.0,0.0,0.0,0.0,0,0,"b'#61359, balance, credit'",0.0,0.0,0,0.0,0.0,0,1736936474000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#admin =  imported["mt5admin"]()
imported["admin_disconnect"]()

In [4]:
df = mt5_deal_df

# Преобразование байтовых строк в обычные строки
df['Comment (|S128)'] = df['Comment (|S128)'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# Функция для извлечения числа между # и ,
def extract_number(comment):
    match = re.search(r'#(\d+),', comment)
    return int(match.group(1)) if match else np.nan

# Функция для извлечения текста после числа
def extract_text(comment):
    match = re.search(r'#\d+,(.*)', comment)
    return match.group(1).strip() if match else np.nan

# Создание первого DataFrame, отфильтрованного по принципу: Action (uint32) = 0 или 1
df1 = df[df['Action (uint32)'].isin([0, 1])].copy()
df1['Number'] = df1['Comment (|S128)'].apply(extract_number)
df1['Text'] = df1['Comment (|S128)'].apply(extract_text)

# Создание второго DataFrame, отфильтрованного по принципу: Action (uint32) > 1
df2 = df[df['Action (uint32)'] > 1].copy()
df2['Number'] = df2['Comment (|S128)'].apply(extract_number)
df2['Text'] = df2['Comment (|S128)'].apply(extract_text)

df3 = df2[df2['Text'].str.contains('balance', case=False, na=False) & ~df2['Text'].str.contains('referrals', case=False, na=False)]
mt_balans_id_list = df3['Number'].dropna().astype(int).tolist()
print("\n", len(mt_balans_id_list), "mt_balans_id_list = ", "mt_balans_id_list")
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\mt_balans_id_list.csv"
text = "БАЛАНСОВЫЕ операции на MT5:"
imported["int_list_to_csv_file"](mt_balans_id_list, file_path, text)
imported["list_print"](mt_balans_id_list, text)
imported["pd_set_option"](text, df3, 3)

df5 = df2[df2['Text'].str.contains('balance', case=False, na=False) & df2['Text'].str.contains('referrals', case=False, na=False)]
mt_referrals_id_list = df5['Number'].dropna().astype(int).tolist()
print("\n",len(mt_referrals_id_list), "mt_referrals_id_list = ", "mt_referrals_id_list")
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\mt_referrals_id_list.csv"
text = "РЕФЕРАЛЬНЫЕ операции на МТ5:"
imported["int_list_to_csv_file"](mt_referrals_id_list, file_path, text)
imported["list_print"](mt_referrals_id_list, text)
imported["pd_set_option"](text, df5, 3)

df4 = df1[df1['Text'].str.contains('RP', case=False, na=False) & df1['Text'].str.contains('SP', case=False, na=False)]
mt_trade_id_list = df4['Number'].dropna().astype(int).tolist()
print("\n",len(mt_trade_id_list), "mt_trade_id_list = ", "mt_trade_id_list")
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\mt_trade_id_list.csv"
text = "ТОРГОВЫЕ операции на МТ5:"
imported["int_list_to_csv_file"](mt_trade_id_list, file_path, text)
imported["list_print"](mt_trade_id_list, text)
imported["pd_set_option"](text, df4, 3)




 7990 mt_balans_id_list =  mt_balans_id_list
Путь к сохранённому списку [БАЛАНСОВЫЕ операции на MT5:]: [C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\mt_balans_id_list.csv]
[7990] элементов в списке [БАЛАНСОВЫЕ операции на MT5:] список: [42623, 42625, 42626, 42627, 42630, 42631, 42632, 42634, 42635, 42650, 42651, 42659, 42661, 42662, 42669, 42673, 42674, 42681, 42682, 42684, 42685, 42686, 42687, 42688, 42692, 42693, 42702, 42704, 42715, 42720, 42726, 42727, 42740, 42741, 42745, 42747, 42748, 42751, 42752, 42754, 42755, 42759, 42760, 42763, 42769, 42790, 42791, 42794, 42801, 42802, 42833, 42834, 42598, 42605, 42606, 42613, 42614, 42621, 42622, 42633, 42643, 42645, 42646, 42654, 42655, 42656, 42657, 42658, 42663, 42664, 42667, 42668, 42670, 42671, 42672, 42698, 42700, 42707, 42713, 42753, 42762, 42764, 42773, 42774, 42777, 42789, 42792, 42793, 42808, 42809, 42812, 42814, 42816, 42819, 42820, 42821, 42827, 42837, 42838, 42843, 42844, 42845, 42846,

,Deal (uint64),ExternalID (|S128),Login (uint64),Dealer (uint64),Order (uint64),Action (uint32),Entry (uint32),Digits (uint32),DigitsCurrency (uint32),ContractSize (float64),Time (int64),Symbol (|S128),Price (float64),Volume (uint64),Profit (float64),Storage (float64),Commission (float64),ObsoleteValue (float64),RateProfit (float64),RateMargin (float64),ExpertID (uint64),PositionID (uint64),Comment (|S128),ProfitRaw (float64),PricePosition (float64),VolumeClosed (uint64),TickValue (float64),TickSize (float64),Flags (uint64),TimeMsc (int64),Reason (uint32),Gateway (|S64),PriceGateway (float64),ModificationFlags (uint32),PriceSL (float64),PriceTP (float64),VolumeExt (uint64),VolumeClosedExt (uint64),Fee (float64),Value (float64),MarketBid (float64),MarketAsk (float64),MarketLast (float64),Number,Text
0,3436674,b'',139807,1100,0,2,0,2,2,0.0,1688983353,b'',0.0,0,1000.0,0.0,0.0,0.0,0.0,0.0,0,0,"#42623, balance, deposit",0.0,0.0,0,0.0,0.0,0,1688983353000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,42623,"balance, deposit"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268708,3709172,b'',263659,1100,0,3,0,2,2,0.0,1736936474,b'',0.0,0,-259.0,0.0,0.0,0.0,0.0,0.0,0,0,"#61359, balance, credit",0.0,0.0,0,0.0,0.0,0,1736936474000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,61359,"balance, credit"



 192 mt_referrals_id_list =  mt_referrals_id_list
Путь к сохранённому списку [РЕФЕРАЛЬНЫЕ операции на МТ5:]: [C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\mt_referrals_id_list.csv]
[192] элементов в списке [РЕФЕРАЛЬНЫЕ операции на МТ5:] список: [1, 2, 3, 5, 6, 7, 9, 8, 12, 13, 11, 15, 17, 22, 16, 19, 20, 21, 23, 24, 25, 27, 28, 35, 38, 31, 33, 34, 36, 39, 40, 41, 43, 44, 45, 46, 47, 49, 64, 50, 51, 52, 53, 54, 56, 57, 58, 59, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 92, 89, 91, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 90, 117, 129, 115, 116, 118, 119, 120, 121, 122, 123, 127, 128, 130, 131, 132, 133, 135, 136, 137, 138, 139, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 124, 125, 126, 140, 150, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177

,Deal (uint64),ExternalID (|S128),Login (uint64),Dealer (uint64),Order (uint64),Action (uint32),Entry (uint32),Digits (uint32),DigitsCurrency (uint32),ContractSize (float64),Time (int64),Symbol (|S128),Price (float64),Volume (uint64),Profit (float64),Storage (float64),Commission (float64),ObsoleteValue (float64),RateProfit (float64),RateMargin (float64),ExpertID (uint64),PositionID (uint64),Comment (|S128),ProfitRaw (float64),PricePosition (float64),VolumeClosed (uint64),TickValue (float64),TickSize (float64),Flags (uint64),TimeMsc (int64),Reason (uint32),Gateway (|S64),PriceGateway (float64),ModificationFlags (uint32),PriceSL (float64),PriceTP (float64),VolumeExt (uint64),VolumeClosedExt (uint64),Fee (float64),Value (float64),MarketBid (float64),MarketAsk (float64),MarketLast (float64),Number,Text
7638,3441739,b'',127717,1100,0,5,0,2,2,0.0,1694706001,b'',0.0,0,195.00,0.0,0.0,0.0,0.0,0.0,0,0,"#1, balance, referrals",0.0,0.0,0,0.0,0.0,0,1694706001000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,1,"balance, referrals"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268632,3709096,b'',237728,1100,0,5,0,2,2,0.0,1736745695,b'',0.0,0,786.43,0.0,0.0,0.0,0.0,0.0,0,0,"#205, balance, referrals",0.0,0.0,0,0.0,0.0,0,1736745695000,2,b'',0.0,32,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,205,"balance, referrals"



 130263 mt_trade_id_list =  mt_trade_id_list
Путь к сохранённому списку [ТОРГОВЫЕ операции на МТ5:]: [C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\mt_trade_id_list.csv]
[130263] элементов в списке [ТОРГОВЫЕ операции на МТ5:] список: [906, 907, 908, 910, 911, 914, 953, 954, 955, 957, 960, 971, 972, 973, 974, 975, 976, 982, 984, 985, 996, 997, 998, 999, 1000, 1020, 1021, 1029, 1032, 1033, 1035, 1037, 1050, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1078, 1080, 1104, 1105, 1106, 1126, 1134, 1135, 1137, 1138, 1139, 1141, 1142, 1194, 1212, 1213, 1214, 1220, 1222, 1224, 1226, 1228, 1229, 1239, 1243, 1245, 1246, 1247, 1266, 1267, 1268, 1269, 1270, 1274, 1275, 1276, 1287, 1288, 1289, 1291, 1292, 1293, 1309, 1343, 1358, 1359, 1360, 1361, 1366, 1385, 1389, 1396, 1397, 1399, 1401, 1404, 1407, 1415, 1416, 1417, 1423, 1425, 1427, 1428, 1465, 1470, 1471, 1472, 1489, 1490, 1491, 1508, 1509, 1517, 1518, 1519, 1520, 1521, 1522, 1523, 1524, 1525, 1527, 154

,Deal (uint64),ExternalID (|S128),Login (uint64),Dealer (uint64),Order (uint64),Action (uint32),Entry (uint32),Digits (uint32),DigitsCurrency (uint32),ContractSize (float64),Time (int64),Symbol (|S128),Price (float64),Volume (uint64),Profit (float64),Storage (float64),Commission (float64),ObsoleteValue (float64),RateProfit (float64),RateMargin (float64),ExpertID (uint64),PositionID (uint64),Comment (|S128),ProfitRaw (float64),PricePosition (float64),VolumeClosed (uint64),TickValue (float64),TickSize (float64),Flags (uint64),TimeMsc (int64),Reason (uint32),Gateway (|S64),PriceGateway (float64),ModificationFlags (uint32),PriceSL (float64),PriceTP (float64),VolumeExt (uint64),VolumeClosedExt (uint64),Fee (float64),Value (float64),MarketBid (float64),MarketAsk (float64),MarketLast (float64),Number,Text
2,3436694,b'',90054,0,0,0,0,5,2,100000.0,1689000332,b'AUDCHF',0.59111,300,0.0,0.0,0.0,0.0,1.091238,0.61471,0,4241640,"#906, RP=1.13, SP=-3.38",0.0,0.0,0,0.0,0.0,0,1689000332337,0,b'',0.0,256,0.0,0.0,3000000,0,0.0,0.0,0.56264,0.56374,0.0,906.0,"RP=1.13, SP=-3.38"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268705,3709169,b'',262078,0,0,1,0,4,2,10000.0,1736758366,b'Bitwise_LTD',6.35600,300,0.0,0.0,0.0,0.0,1.000000,1.00000,0,4327011,"#200971, RP=1.0, SP=0.0",0.0,0.0,0,0.0,0.0,0,1736758366086,0,b'',0.0,256,0.0,0.0,3000000,0,0.0,0.0,7.25450,7.31780,0.0,200971.0,"RP=1.0, SP=0.0"


In [5]:
crm_balans_id_list = imported["load_account_list"](r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250118\crm_balans_id_list_250118.csv")
imported["list_print"](crm_balans_id_list, "ID БАЛАНСОВЫХ операций в ЦРМ")

crm_referrals_id_list = imported["load_account_list"](r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250118\crm_referrals_id_list_250118.csv")
imported["list_print"](crm_referrals_id_list, "ID РЕФЕРАЛЬНЫХ операций в ЦРМ")

crm_statement_id_list = imported["load_account_list"](r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250118\crm_statement_id_list_250118.csv")
imported["list_print"](crm_statement_id_list, "ID ТОРГОВЫХ операций в ЦРМ")

Файл: C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250118\crm_balans_id_list_250118.csv; Длинна списка: 8158
[8158] элементов в списке [ID БАЛАНСОВЫХ операций в ЦРМ] список: [42598, 42605, 42606, 42613, 42614, 42621, 42622, 42623, 42625, 42626, 42627, 42630, 42631, 42632, 42633, 42634, 42635, 42643, 42645, 42646, 42650, 42651, 42654, 42655, 42656, 42657, 42658, 42659, 42661, 42662, 42663, 42664, 42667, 42668, 42669, 42670, 42671, 42672, 42673, 42674, 42681, 42682, 42684, 42685, 42686, 42687, 42688, 42692, 42693, 42698, 42700, 42702, 42704, 42707, 42713, 42715, 42720, 42726, 42727, 42740, 42741, 42745, 42747, 42748, 42751, 42752, 42753, 42754, 42755, 42759, 42760, 42762, 42763, 42764, 42769, 42773, 42774, 42777, 42789, 42790, 42791, 42792, 42793, 42794, 42801, 42802, 42808, 42809, 42812, 42814, 42816, 42819, 42820, 42821, 42827, 42833, 42834, 42837, 42838, 42843, 42844, 42845, 42846, 42847, 42848, 42849, 42851, 42857, 42858, 42859, 42860, 42861, 42863,

In [7]:
dif_balans_id_list      = set(crm_balans_id_list)    - set(mt_balans_id_list)
text = "dif_balans_id_list"
imported["list_print"](dif_balans_id_list,  text)
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\dif_balans_id_list.csv"
imported["int_list_to_csv_file"](dif_balans_id_list, file_path, text)

dif_referrals_id_list   = set(crm_referrals_id_list) - set(mt_referrals_id_list)
text = "dif_referrals_id_list"
imported["list_print"](dif_referrals_id_list,  text)
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\dif_referrals_id_list.csv"
imported["int_list_to_csv_file"](dif_referrals_id_list, file_path, text)

dif_statement_id_list   = set(crm_statement_id_list) - set(mt_trade_id_list)
text = "dif_statement_id_list"
imported["list_print"](dif_statement_id_list,  text)
file_path = r"C:\unique_data\rep_fo_metatrader_server\files\migration_errors\migration_errors_250119\dif_trade_id_list.csv"
imported["int_list_to_csv_file"](dif_statement_id_list, file_path, text)

[175] элементов в списке [dif_balans_id_list] список: {61440, 50177, 61442, 52227, 61443, 61445, 61446, 61447, 61448, 61449, 60938, 61451, 60940, 61454, 61456, 60461, 55854, 60466, 47679, 47168, 47169, 47681, 52800, 52802, 54338, 58945, 49225, 56411, 54883, 56419, 55403, 55405, 57966, 58478, 57968, 57973, 58487, 50302, 46724, 46725, 46726, 57501, 58543, 51907, 51911, 54983, 48338, 48855, 48856, 52456, 55535, 55536, 54009, 52483, 54024, 61225, 47938, 47939, 61250, 61252, 59718, 61256, 61259, 52044, 52045, 59726, 61263, 61264, 61265, 59730, 59732, 59734, 61270, 61275, 61276, 61280, 61281, 61283, 61285, 61286, 61292, 61295, 61296, 58737, 61300, 56181, 61305, 61307, 61311, 61312, 56193, 58242, 58245, 58246, 61318, 51593, 61321, 61324, 50062, 58254, 58255, 61326, 58260, 61342, 56735, 50594, 52130, 61346, 58789, 61351, 61352, 52137, 61355, 54188, 61364, 52151, 61368, 61369, 61370, 61371, 57276, 61376, 61377, 61378, 61379, 61380, 61381, 61382, 61383, 61385, 61386, 61390, 61391, 61392, 61393, 